# CANDIDATE FEATURE IDENTIFICATION
The purpose of this notebook is to aggregate in one location the potential features to include in the training set. 
Inputs: External Dataset Parquet Files. 
Desired Outputs: 
- Of redundant variables, select the most reliable and informative one. 
- Understand how to calculate any computed features.

Notes
- Use a separate .py module under src/external_dataset_name/ingest.py to pull in the data with API or other method and write to data/interim/external_dataset_name.parquet. 
- This notebook should read in just a small sample of each dataset for feature exploration. Reading in the full dataset may crash the Jupyter kernel. 

For each feature, attempt to identify:
- time of observation
- unit of observation / what it actually is
- geographic scope (i.e. is it at the census block group, tract, or zip code level?)
- missingness
- what preprocessing is required (e.g. geocoding to census block group, converting a raw count to density)

In [2]:
from pathlib import Path

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

In [3]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

## Lottery and Sales Data (for reference when merging)

In [4]:
lottery_locations_file_path = Path(".").resolve().parent / "data/final/location_data.parquet"

#locations_pq = pq.ParquetFile(lottery_locations_file_path)
#locations_first_100_rows = next(locations_pq.iter_batches(batch_size=100))
#locations_df = pa.Table.from_batches([locations_first_100_rows]).to_pandas()

locations_df = pd.read_parquet(str(lottery_locations_file_path))

In [5]:
print(locations_df.shape)

locations_df.head()

(16789, 14)


,agtno,busnm,busaddr,buscity,buszip,chain,bustype,longitude,latitude,geometry,bg_geoid,tract_geoid,region,competitors_within_1_mi
0,55194,BOWL-O-DROME,2712 PINE AVE,NIAGARA FALLS,14301,independent,BOWLING CENTERS,-79.02878,43.09513,b'\x01\x01\x00\x00\x00\xd9\xb1\x11\x88\xd7\xc1S\xc0*oG8-\x8cE@',360630210002,36063021000,Western New York,15
1,68523,YAHIR & SISTERS GROCERY CORP,165 MCLEAN AVE,YONKERS,10705,independent,GROCERY STORE,-73.88996,40.91546,b'\x01\x01\x00\x00\x00\xe2\xaf\xc9\x1a\xf5xR\xc0\xe9e\x14\xcb-uD@',361190013013,36119001301,Mid-Hudson,44
2,85214,HONG KONG SUPERMARKET,37-11 MAIN STREET,FLUSHING,11354,independent,SUPERMARKETS,-73.83094,40.76119,b'\x01\x01\x00\x00\x00\x0c<\xf7\x1e.uR\xc0h\x05\x86\xacnaD@',360810871002,36081087100,New York City,103
3,105306,MOBIL MART,380 COMMACK ROAD,DEER PARK,11729,independent,CONV. STORE,-73.31394,40.75946,b'\x01\x01\x00\x00\x00\x00:\xcc\x97\x17TR\xc0aO;\xfc5aD@',361031228012,36103122801,Long Island,33
4,108863,FQ MULTI-SERVICE,1702 PALMETTO STREET,RIDGEWOOD,11385,independent,GROCERY STORE,-73.90473,40.70571,b'\x01\x01\x00\x00\x00rm\xa8\x18\xe7yR\xc0\xe7:\x8d\xb4TZD@',360810589003,36081058900,New York City,159


In [6]:
lottery_sales_file_path = Path(".").resolve().parent / "data/final/sales_data.parquet"

#sales_pq = pq.ParquetFile(lottery_sales_file_path)
#sales_first_100_rows = next(sales_pq.iter_batches(batch_size=1000))
#sales_df = pa.Table.from_batches([sales_first_100_rows]).to_pandas()

sales_df = pd.read_parquet(str(lottery_sales_file_path))

In [7]:
print(sales_df.shape)

sales_df.head()

(14351, 15)


,agtno,total_sales,first_date,last_date,first_pos_date,last_pos_date,pos_sales_days,num_cal_days,num_active_days,days_before_first_sale,days_after_last_sale,avg_cal_daily_sales,avg_active_daily_sales,use_active_period,avg_daily_sales
0,100001,475315.5,2025-01-01,2025-12-31,2025-01-01,2025-12-31,365.0,365,365.0,0.0,0.0,1302.234247,1302.234247,False,1302.234247
1,100005,1857239.5,2025-01-01,2025-12-31,2025-01-01,2025-12-31,365.0,365,365.0,0.0,0.0,5088.327397,5088.327397,False,5088.327397
2,100006,1091206.0,2025-01-01,2025-12-31,2025-01-01,2025-12-31,362.0,365,365.0,0.0,0.0,2989.605479,2989.605479,False,2989.605479
3,100009,854795.0,2025-01-01,2025-12-31,2025-01-01,2025-12-31,365.0,365,365.0,0.0,0.0,2341.904110,2341.904110,False,2341.904110
4,100013,958174.0,2025-01-01,2025-12-30,2025-01-01,2025-08-19,231.0,364,231.0,0.0,133.0,2632.346154,4147.939394,True,4147.939394


## Overture Maps Data

In [8]:
interim_overture_file_path = Path(".").resolve().parent / "data/interim/external/overture_filtered.parquet"

#interim_overture_pq = pq.ParquetFile(interim_overture_file_path)
#interim_overture_first_100_rows = next(interim_overture_pq.iter_batches(batch_size=5000))
#interim_overture_df = pa.Table.from_batches([interim_overture_first_100_rows]).to_pandas()

interim_overture_df = pd.read_parquet(interim_overture_file_path)

print(interim_overture_df.shape)
interim_overture_df.sample(5)

(393891, 10)


,id,geometry,longitude,latitude,basic_category,taxonomy_primary,taxonomy_hierarchy,confidence,brand_name,first_seen
73965,6b75a5b3-6c67-4bf0-9d8e-d34fa9c7fb31,b'\x01\x01\x00\x00\x00WI\x1a\x92\t~R\xc0\xe1R!`\xa5aD@',-73.969334,40.762859,financial_service,investing,"[services_and_business, financial_service, investing]",0.919912,NaN,2025-06-25.0
270715,a4aedeb5-a5e8-43d9-9828-bee9906d786c,b'\x01\x01\x00\x00\x00\x94\xe3\xfa\x1a$\xbdR\xc0.\x83~\xeb\xb8OF@',-74.955329,44.622831,restaurant,bar_and_grill_restaurant,"[food_and_drink, restaurant, bar_and_grill_restaurant]",0.990219,NaN,2025-06-25.0
75696,d1a4e55d-796d-4db8-b074-d5e19c5b0b32,b'\x01\x01\x00\x00\x00;\x02\x97KSzR\xc0\x0e\xedJh\x9eaD@',-73.911334,40.762647,home_service,contractor,"[services_and_business, home_service, contractor]",0.919912,NaN,2025-06-25.0
393831,00c9a14a-cb5c-4c4c-b944-2c470f740817,b'\x01\x01\x00\x00\x00\x86\xeb\xacq\x91eS\xc0\xdb8\xbd\xe6\xb0\x98E@',-77.587002,43.192899,specialized_health_care,neurology,"[health_care, outpatient_care_facility, specialized_health_care, neurology]",0.919912,NaN,2025-06-25.0
15281,04b8845d-dcb9-4389-b217-4af0cb03481b,b'\x01\x01\x00\x00\x00Q\xe2\x8c\xc4xqR\xc0W~\xd7o\xbf\x8aE@',-73.772996,43.083967,travel_service,travel_service,"[travel_and_transportation, travel_service]",0.919912,NaN,2025-06-25.0


In [9]:
overture_file_path = Path(".").resolve().parent / "data/interim/external/merged_with_loc/location_data_overture.parquet"

#overture_pq = pq.ParquetFile(overture_file_path)
#overture_first_100_rows = next(overture_pq.iter_batches(batch_size=100))
#overture_df = pa.Table.from_batches([overture_first_100_rows]).to_pandas()

overture_df = pd.read_parquet(overture_file_path)

print(overture_df.shape)
overture_df.sample(5)

(16740, 14)


,agtno,busnm,busaddr,buscity,buszip,chain,bustype,longitude,latitude,geometry,bg_geoid,tract_geoid,commercial_places_within_1_mi,community_places_within_1_mi
16086,116275,CUSE GAS & GO INC,200 N STATE ST,SYRACUSE,13203,0,CONV. STORE,-76.14839,43.05283,"b'\x01\x01\x00\x00\x00lC\xc58\x7f\tS\xc0\xb4\x1f)""\xc3\x86E@'",360670023002,36067002300,466,217
1680,94965,7-ELEVEN #34495A,395 FLATBUSH AVENUE EXTENSION,BROOKLYN,11201,30,CONV. STORE,-73.98110,40.68934,b'\x01\x01\x00\x00\x00\xc2\x86\xa7W\xca~R\xc0\x8e\xe9\tK<XD@',360470033002,36047003300,1918,586
14332,113856,RJSZ,706 W 181ST STREET,NEW YORK,10033,0,NEWS-SMOKE,-73.93614,40.85032,"b""\x01\x01\x00\x00\x00\x8e\x1e\xbf\xb7\xe9{R\xc0?\x91'I\xd7lD@""",360610271004,36061027100,613,201
14086,113438,DA GOOD LIFE INC,401 JERICHO TPKE,NEW HYDE PARK,11040,0,CONV. STORE,-73.69012,40.73158,b'\x01\x01\x00\x00\x001\x94\x13\xed*lR\xc0/4\xd7i\xa4]D@',360593030005,36059303000,256,55
1325,99311,RITE AID #10585,2064 MILL AVE,BROOKLYN,11234,296,DRUG STORE,-73.91453,40.61558,b'\x01\x01\x00\x00\x00\x7fM\xd6\xa8\x87zR\xc0-\tPS\xcbND@',360470686001,36047068600,342,50


## LEHD Data

In [10]:
rac_file_path = Path(".").resolve().parent / "data/interim/external/merged_with_loc/location_data_rac.parquet"

#rac_pq = pq.ParquetFile(rac_file_path)
#rac_first_100_rows = next(rac_pq.iter_batches(batch_size=100))
#rac_df = pa.Table.from_batches([rac_first_100_rows]).to_pandas()

rac_df = pd.read_parquet(rac_file_path)

In [11]:
print(rac_df.shape)
rac_df.sample(2)

(16790, 20)


,agtno,busnm,busaddr,buscity,buszip,chain,bustype,longitude,latitude,geometry,bg_geoid,tract_geoid,region,competitors_within_1_mi,rac_c000,rac_ce01,rac_ce02,rac_ce03,rac_ce01_prop,rac_ce02_prop
8315,85741,BOLLA OPERATING CORP STORE #15,135-45 CROSSBAY BLVD,OZONE PARK,11417,chain,CONV. STORE,-73.84291,40.67266,b'\x01\x01\x00\x00\x00+\xde\xc8<\xf2uR\xc0\xe7\xa9\x0e\xb9\x19VD@',360810088001,36081008800,New York City,54,788.0,125.0,214.0,449.0,0.158629,0.271574
2582,112934,IMPERIAL DELI MART,111 CRESCENT PL,YONKERS,10704,independent,CONV. STORE,-73.84892,40.91918,b'\x01\x01\x00\x00\x00\xe7:\x8d\xb4TvR\xc0\x92\x91\xb3\xb0\xa7uD@',361190024031,36119002403,Mid-Hudson,62,656.0,91.0,135.0,430.0,0.138720,0.205793


In [12]:
wac_file_path = Path(".").resolve().parent / "data/interim/external/merged_with_loc/location_data_wac.parquet"

#wac_pq = pq.ParquetFile(wac_file_path)
#wac_first_100_rows = next(wac_pq.iter_batches(batch_size=100))
#wac_df = pa.Table.from_batches([wac_first_100_rows]).to_pandas()

wac_df = pd.read_parquet(wac_file_path)

In [13]:
print(wac_df.shape)
wac_df.sample(2)

(16793, 39)


,agtno,busnm,busaddr,buscity,buszip,chain,bustype,longitude,latitude,geometry,bg_geoid,tract_geoid,competitors_within_1_mi,wac_cns01,wac_cns02,wac_cns03,wac_cns04,wac_cns05,wac_cns06,wac_cns07,wac_cns08,wac_cns09,wac_cns10,wac_cns11,wac_cns12,wac_cns13,wac_cns14,wac_cns15,wac_cns16,wac_cns17,wac_cns18,wac_cns19,wac_cns20,wac_c000,wac_ce01,wac_ce02,wac_ce03,wac_ce01_prop,wac_ce02_prop
5236,47748,HANNAFORD #8370,50 KELLOGG RD,NEW HARTFORD,13413,chain,SUPERMARKETS,-75.27758,43.05383,b'\x01\x01\x00\x00\x00\x84\x81\xe7\xde\xc3\xd1R\xc0\x97\xc5\xc4\xe6\xe3\x86E@',360650253005,36065025300,3,0.0,0.0,0.0,27.0,52.0,23.0,182.0,25.0,0.0,21.0,0.0,6.0,0.0,38.0,0.0,98.0,0.0,19.0,71.0,0.0,562.0,154.0,183.0,225.0,0.274021,0.325623
14519,114082,7-ELEVEN #32118D,146-26 NORTHERN BLVD,FLUSHING,11354,chain,CONV. STORE,-73.81974,40.76511,b'\x01\x01\x00\x00\x00F\xce\xc2\x9evtR\xc0\xd9\xeb\xdd\x1f\xefaD@',360811163012,36081116301,108,0.0,0.0,0.0,7.0,0.0,0.0,36.0,3.0,21.0,25.0,4.0,2.0,2.0,5.0,2.0,176.0,0.0,27.0,11.0,0.0,321.0,44.0,80.0,197.0,0.137072,0.249221


## AADT Data

In [14]:
#aadt_file_path = Path(".").resolve().parent / "data/interim/external/merged_with_loc/data_aadt_2025.parquet"

#aadt_pq = pq.ParquetFile(aadt_file_path)
#aadt_first_100_rows = next(aadt_pq.iter_batches(batch_size=100))
#aadt_df = pa.Table.from_batches([aadt_first_100_rows]).to_pandas()

aadt_file_path = Path(".").resolve().parent / "data/interim/external/merged_with_loc/data_aadt_2025.parquet"
aadt_df = pd.read_parquet(str(aadt_file_path))

In [15]:
print(aadt_df['aadt_count'].isna().sum(), len(aadt_df))
print(aadt_df['aadt_road_name'].nunique())
print(aadt_df['aadt_count'].describe())
aadt_df.sample(10)

33 16793
2681
count     16760.000000
mean      12065.746539
std       11110.397234
min           2.000000
25%        5420.500000
50%        9633.000000
75%       15579.000000
max      199411.000000
Name: aadt_count, dtype: float64


,agtno,busnm,busaddr,buscity,buszip,chain,bustype,longitude,latitude,bg_geoid,tract_geoid,competitors_within_1_mi,index_right,aadt_count,aadt_road_name,aadt_functional_class,aadt_lanes,Rmp,dist_to_read_m
2890,111565,WALMART SUPERCENTER #2264,1320 ALTAMONT AVE,SCHENECTADY,12303,chain,DEPARTMENT,-73.94191,42.78592,360930329025,36093032902,24,17221.0,18572.0,Altamont Ave,14,6.0,,78.366475
5072,55905,BAP GROUP/ALLENTOWN TRADING,553 DELAWARE AVE,BUFFALO,14202,chain,CONV. STORE,-78.87405,42.89925,360290068023,36029006802,38,32596.0,13929.0,DELAWARE AVE,14,2.0,,4.298665
2737,114063,THE RED SEA FOOD MART INC,3154 BAILEY AVE,BUFFALO,14215,independent,CONV. STORE,-78.81366,42.94071,360290047012,36029004701,32,32788.0,18930.0,BAILEY AVE,14,4.0,,1.551011
14276,28964,A & R GROCERY,59-10 92ND ST,ELMHURST,11373,independent,CONV. STORE,-73.86912,40.73397,360810683003,36081068300,132,8526.0,3647.0,59TH AVE,17,2.0,,18.750035
15100,38617,PILOT TRAVEL CENTER #0494,1128 DUANESBURG RD,SCHENECTADY,12306,chain,TRUCKSTOPS,-74.02870,42.78136,360930326011,36093032601,2,17448.0,2688.0,,07,2.0,,58.552873
4260,55265,KENYON'S VARIETY,9700 TRANSIT RD,EAST AMHERST,14051,independent,CONV. STORE,-78.69779,43.03390,360290090042,36029009004,2,34793.0,15425.0,NORTH FRENCH RD,14,2.0,,52.787511
16673,114160,LOCOMOTIONS SPORTS BAR AND GRI,136 RAILROAD AVE,COBLESKILL,12043,independent,RESTAURANT-HOTEL,-74.48355,42.67701,360957402005,36095740200,7,52555.0,234.0,RAILROAD AVE,19,2.0,,0.227690
6467,87496,BAY STREET BAZAAR INC,543 BAY STREET,STATEN ISLAND,10304,independent,DELI-GROCERY,-74.07648,40.62861,360850021002,36085002100,34,9246.0,11881.0,BAY ST,14,2.0,,0.508937
2450,116416,FATBOY DELI AND GRILL 1 CORP,17222 HILLSIDE AVE,JAMAICA,11432,independent,DELI-GROCERY,-73.78952,40.71195,360810462002,36081046200,80,6990.0,19937.0,HILLSIDE AVE,14,4.0,,4.189887
8253,112082,QUENCH LIQUORS,90 W MAIN ST,PATCHOGUE,11772,independent,LIQUOR,-73.01791,40.76532,361031589021,36103158902,22,11127.0,9795.0,W MAIN ST,16,2.0,,60.234227


## EPA Data

In [16]:
epa_file_path = Path(".").resolve().parent / "data/interim/external/merged_with_loc/location_data_epa.parquet"

#epa_pq = pq.ParquetFile(epa_file_path)
#epa_first_100_rows = next(epa_pq.iter_batches(batch_size=100))
#epa_df = pa.Table.from_batches([epa_first_100_rows]).to_pandas()

epa_df = pd.read_parquet(epa_file_path)

In [17]:
print(epa_df.shape)
epa_df.sample(5)

(13764, 35)


,agtno,busnm,busaddr,buscity,buszip,chain,bustype,longitude,latitude,geometry,bg_geoid,tract_geoid,competitors_within_1_mi,OBJECTID,STATEFP,COUNTYFP,TRACTCE,BLKGRPCE,D1A,D1B,D1C,D1C8_RET,D1C8_OFF,D1C8_IND,D1C8_SVC,D1C8_ENT,D1C8_ED,D1C8_HLTH,D1C8_PUB,D3A,NatWalkInd,STATEFP2,COUNTYFP2,TRACTCE2,BLKGRPCE2
10846,111775,NEWBRIDGE PETROLEUM CORP,671 NEWBRIDGE RD,EAST MEADOW,11554,independent,CONV. STORE,-73.54404,40.70655,b'\x01\x01\x00\x00\x00\xd0\xed%\x8d\xd1bR\xc0\x91~\xfb:pZD@',360594083005,36059408300,16,162834,36,59,408300,5,4.365466,12.266244,5.181307,0.200382,0.128817,1.187979,3.234739,0.229008,0.014313,0.186069,0.0,23.427784,16.166667,36,059,408300,5
4668,57904,KIEBZAK'S LTD,38 CROCKER ST,BUFFALO,14212,independent,RESTAURANT-HOTEL,-78.78843,42.89344,b'\x01\x01\x00\x00\x00\xdcK\x1a\xa3u\xb2S\xc0\x18x\xee=\\rE@',360290099002,36029009900,11,160085,36,29,9900,2,1.051577,2.244939,1.087023,0.102401,0.000000,0.287510,0.614404,0.051200,0.000000,0.031508,0.0,18.719715,13.000000,36,029,009900,2
2794,113960,SAYONA CARD AND GIFT INC,644 MIDDLE COUNTRY RD,SELDEN,11784,independent,STATIONERY-CARD,-73.03577,40.86648,b'\x01\x01\x00\x00\x00c\x0bA\x0eJBR\xc0\xadQ\x0f\xd1\xe8nD@',361031581121,36103158112,18,161495,36,103,158112,1,1.744492,5.638126,0.482583,0.161860,0.000000,0.128889,0.122894,0.000000,0.000000,0.068940,0.0,14.841228,7.833333,36,103,158112,1
8149,102158,7-ELEVEN #37018A,500 ATLANTIC AVE,FREEPORT,11520,chain,CONV. STORE,-73.59759,40.64327,b'\x01\x01\x00\x00\x00\xaa\x9a \xea>fR\xc0\xbc?\xde\xabVRD@',360594143032,36059414303,42,153842,36,59,414303,2,4.661548,14.249274,0.407122,0.081424,0.061068,0.020356,0.020356,0.000000,0.000000,0.223917,0.0,33.093689,14.500000,36,059,414303,2
8211,102450,KWIK FILL #403,21 COUNTY ROUTE 115,LINDLEY,14858,chain,CONV. STORE,-77.12715,42.00075,b'\x01\x01\x00\x00\x00\xee\xeb\xc09#HS\xc0j\xbct\x93\x18\x00E@',361019630004,36101963000,0,154989,36,101,963000,4,0.029933,0.068002,0.002389,0.000523,0.000000,0.000970,0.000373,0.000523,0.000000,0.000000,0.0,2.856818,3.833333,36,101,963000,4


## ACS Data

In [18]:
acs_file_path = Path(".").resolve().parent / "data/interim/external/merged_with_loc/location_data_acs.parquet"

#acs_pq = pq.ParquetFile(acs_file_path)
#acs_first_100_rows = next(acs_pq.iter_batches(batch_size=100))
#acs_df = pa.Table.from_batches([acs_first_100_rows]).to_pandas()

acs_df = pd.read_parquet(acs_file_path)

In [19]:
print(acs_df.shape)
acs_df.sample(5)

(16801, 29)


,agtno,busnm,busaddr,buscity,buszip,chain,bustype,longitude,latitude,geometry,bg_geoid,tract_geoid,median_household_income,households_total,total_population,age_18_29_proportion,age_30_64_proportion,age_65_plus_proportion,poverty_rate,bachelors_degree_proportion_25plus,post_grad_degree_proportion_25plus,labor_force_participation_rate,unemployment_rate,owner_occupied_proportion,renter_occupied_proportion,family_households_proportion,nonfamily_multiperson_households_proportion,living_alone_households_proportion,no_vehicle_households_proportion
7763,77893,CFM NO 33-054,1430 N CLINTON AVE,ROCHESTER,14621,0,CONV. STORE,-77.61379,43.18603,"b""\x01\x01\x00\x00\x00('\xdaUHgS\xc0\x96\t\xbf\xd4\xcf\x97E@""",360550047023,36055004702,NaN,237.0,457.0,0.319475,0.557987,0.122538,0.122727,0.093248,0.016077,0.650108,0.093023,0.278481,0.721519,0.738397,0.130802,0.130802,0.666667
2857,100092,ON TRACK NEWS,16 FERRIS AVE,WHITE PLAINS,10601,0,CONV. STORE,-73.77485,41.03337,b'\x01\x01\x00\x00\x00\x8fSt$\x97qR\xc0qU\xd9wE\x84D@',361190090002,36119009000,70750.0,1059.0,2429.0,0.230548,0.647592,0.121861,0.147438,0.155344,0.092637,0.646204,0.081875,0.367328,0.632672,0.724268,0.046270,0.229462,0.154863
9830,101499,DRIFT INN,236 E MAIN ST,EAST ISLIP,11730,0,RESTAURANT-HOTEL,-73.17934,40.73345,"b""\x01\x01\x00\x00\x00V\xb7zNzKR\xc0'\xa0\x89\xb0\xe1]D@""",361031475021,36103147502,161065.0,399.0,1009.0,0.209118,0.455897,0.334985,0.037152,0.249478,0.097077,0.540698,0.000000,1.000000,0.000000,0.679198,0.045113,0.275689,0.000000
6181,64986,JINAL CORP,279 SMITHTOWN BLVD STORE #2,NESCONSET,11767,0,STATIONERY-CARD,-73.14149,40.83272,"b'\x01\x01\x00\x00\x00\x82\xad\x12,\x0eIR\xc0\xd1\\\xa7\x91\x96jD@'",361031354033,36103135403,74412.0,410.0,902.0,0.202882,0.539911,0.257206,0.080548,0.239049,0.136421,0.562819,0.030853,0.741463,0.258537,0.526829,0.104878,0.368293,0.017073
12581,110482,7-ELEVEN STORE #35120,189 MAIN STREET,LEICESTER,14481,321,CONV. STORE,-77.89440,42.77228,b'\x01\x01\x00\x00\x00\xb7b\x7f\xd9=yS\xc0m\xad/\x12\xdabE@',360510308001,36051030800,82457.0,369.0,715.0,0.141259,0.609790,0.248951,0.019394,0.187793,0.120501,0.634014,0.064378,0.815718,0.184282,0.691057,0.035230,0.273713,0.086721


## Master Data Sample

In [20]:
master_file_path = Path(".").resolve().parent / "data/final/master_data.parquet"
master_df = pd.read_parquet(str(master_file_path))

print(master_df.shape)
print(master_df.columns)
master_df.sample(5)

(14340, 58)
Index(['agtno', 'avg_daily_sales', 'chain', 'bustype', 'longitude', 'latitude',
       'bg_geoid', 'tract_geoid', 'region', 'competitors_within_1_mi',
       'aadt_count', 'aadt_functional_class', 'aadt_lanes', 'dist_to_read_m',
       'median_household_income', 'total_population', 'age_18_29_proportion',
       'age_30_64_proportion', 'poverty_rate',
       'bachelors_degree_proportion_25plus',
       'post_grad_degree_proportion_25plus', 'labor_force_participation_rate',
       'unemployment_rate', 'owner_occupied_proportion',
       'family_households_proportion', 'no_vehicle_households_proportion',
       'D1B', 'D1C', 'D3A', 'NatWalkInd', 'commercial_places_within_1_mi',
       'community_places_within_1_mi', 'rac_c000', 'rac_ce01_prop',
       'rac_ce02_prop', 'wac_cns01', 'wac_cns02', 'wac_cns03', 'wac_cns04',
       'wac_cns05', 'wac_cns06', 'wac_cns07', 'wac_cns08', 'wac_cns09',
       'wac_cns10', 'wac_cns11', 'wac_cns12', 'wac_cns13', 'wac_cns14',
       'wac_cns

,agtno,avg_daily_sales,chain,bustype,longitude,latitude,bg_geoid,tract_geoid,region,competitors_within_1_mi,aadt_count,aadt_functional_class,aadt_lanes,dist_to_read_m,median_household_income,total_population,age_18_29_proportion,age_30_64_proportion,poverty_rate,bachelors_degree_proportion_25plus,post_grad_degree_proportion_25plus,labor_force_participation_rate,unemployment_rate,owner_occupied_proportion,family_households_proportion,no_vehicle_households_proportion,D1B,D1C,D3A,NatWalkInd,commercial_places_within_1_mi,community_places_within_1_mi,rac_c000,rac_ce01_prop,rac_ce02_prop,wac_cns01,wac_cns02,wac_cns03,wac_cns04,wac_cns05,wac_cns06,wac_cns07,wac_cns08,wac_cns09,wac_cns10,wac_cns11,wac_cns12,wac_cns13,wac_cns14,wac_cns15,wac_cns16,wac_cns17,wac_cns18,wac_cns19,wac_cns20,wac_c000,wac_ce01_prop,wac_ce02_prop
934,103581,845.949315,independent,CONV. STORE,-73.39283,40.71555,361031232043,36103123204,Long Island,18.0,41317.0,14,11.0,0.582045,NaN,2200.0,0.260455,0.411364,0.105478,0.117735,0.127255,0.474244,0.065530,0.396409,0.512431,0.343923,NaN,NaN,NaN,NaN,77.0,24.0,895.0,0.156425,0.283799,0.0,0.0,0.0,555.0,1453.0,623.0,126.0,74.0,16.0,4.0,19.0,24.0,21.0,306.0,38.0,217.0,3.0,39.0,450.0,0.0,3968.0,0.076109,0.240423
7253,116013,348.464674,independent,CONV. STORE,-73.70050,42.77088,360010128003,36001012800,Capital Region,15.0,1879.0,17,2.0,1.251561,46875.0,733.0,0.366985,0.547067,0.564961,0.044964,0.062950,0.643929,0.105932,0.437500,0.492788,0.060096,25.186425,2.229827,33.466006,16.500000,93.0,37.0,589.0,0.171477,0.297114,0.0,0.0,0.0,1.0,0.0,0.0,54.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,12.0,0.0,9.0,5.0,0.0,82.0,0.280488,0.231707
13226,89103,7951.342466,independent,STATIONERY-CARD,-73.91119,40.88588,360050297002,36005029700,New York City,43.0,4313.0,19,2.0,0.462836,140568.0,1205.0,0.103734,0.625726,0.013803,0.198148,0.474074,0.491539,0.085246,0.516713,0.467967,0.298050,51.515986,12.498691,34.439404,14.833333,324.0,111.0,682.0,0.148094,0.175953,0.0,0.0,0.0,4.0,0.0,30.0,0.0,1.0,0.0,0.0,4.0,1.0,0.0,0.0,20.0,216.0,1.0,24.0,17.0,0.0,318.0,0.220126,0.402516
4824,112893,1066.549315,independent,LIQUOR,-73.86599,41.17269,361190136002,36119013600,Mid-Hudson,13.0,15878.0,14,12.0,0.180468,NaN,1925.0,0.314286,0.612468,0.026305,0.094048,0.102976,0.736199,0.104180,0.349593,0.806504,0.105691,8.429187,3.784370,17.777580,13.166667,126.0,46.0,635.0,0.137008,0.247244,0.0,0.0,0.0,11.0,0.0,0.0,21.0,1.0,0.0,0.0,0.0,16.0,6.0,107.0,318.0,8.0,10.0,10.0,12.0,0.0,520.0,0.100000,0.167308
2938,109364,1272.017808,independent,CONV. STORE,-76.48487,42.48301,361090015001,36109001500,Southern Tier,4.0,21670.0,16,9.0,25.308868,71975.0,2134.0,0.623243,0.258201,0.309113,0.277916,0.491315,0.864840,0.075188,0.189919,0.324932,0.048605,0.310323,0.311216,3.421054,6.166667,87.0,18.0,874.0,0.141876,0.184211,188.0,0.0,0.0,2.0,1294.0,26.0,52.0,36.0,1.0,224.0,23.0,171.0,17.0,10.0,0.0,191.0,0.0,35.0,23.0,29.0,2322.0,0.068045,0.133936


## Missingness and Variance Check

In [21]:
def summarize_numeric_features(df, numeric_cols):

    summary = pd.DataFrame({
        "pct_missing": df[numeric_cols].isna().mean() * 100,
        "pct_zero": (df[numeric_cols] == 0).mean() * 100,
        "pct_mode": [df[col].value_counts(normalize=True, dropna=True).max() * 100 for col in numeric_cols],
        "mode": [df[col].value_counts(dropna=True).idxmax() if df[col].notna().any() else pd.NA for col in numeric_cols],
        "mean": df[numeric_cols].mean(),
        "std": df[numeric_cols].std(),
        "iqr": df[numeric_cols].quantile(0.75) - df[numeric_cols].quantile(0.25),
        "cv": (df[numeric_cols].std()) / (df[numeric_cols].mean().abs()),
        "n_unique": df[numeric_cols].nunique()
    })

    return summary.sort_values("pct_missing", ascending=False)

def summarize_categorical_features(df, categorical_cols):
    results = {}

    for col in categorical_cols:

        pct_missing = df[col].isna().mean() * 100

        distribution = df[col].value_counts(dropna=False, normalize=True).mul(100).rename("pct").to_frame()

        results[col] = {"pct_missing": pct_missing, "distribution": distribution}

    return results

In [22]:
numeric_features = ['competitors_within_1_mi', 'aadt_count', 'aadt_lanes', 'dist_to_read_m',
       'median_household_income', 'total_population', 'age_18_29_proportion', 'age_30_64_proportion',
       'poverty_rate', 'bachelors_degree_proportion_25plus', 'post_grad_degree_proportion_25plus',
       'labor_force_participation_rate', 'unemployment_rate', 'owner_occupied_proportion',
       'family_households_proportion', 'no_vehicle_households_proportion',
       'D1B', 'D1C', 'D3A', 'NatWalkInd', 'commercial_places_within_1_mi',
       'community_places_within_1_mi', 'rac_c000', 'rac_ce01_prop',
       'rac_ce02_prop', 'wac_cns01', 'wac_cns02', 'wac_cns03', 'wac_cns04',
       'wac_cns05', 'wac_cns06', 'wac_cns07', 'wac_cns08', 'wac_cns09',
       'wac_cns10', 'wac_cns11', 'wac_cns12', 'wac_cns13', 'wac_cns14',
       'wac_cns15', 'wac_cns16', 'wac_cns17', 'wac_cns18', 'wac_cns19',
       'wac_cns20', 'wac_c000', 'wac_ce01_prop', 'wac_ce02_prop']

categorical_features = ['chain', 'bustype', 'aadt_functional_class']

numeric_summary = summarize_numeric_features(master_df, numeric_features)
categorical_summary = summarize_categorical_features(master_df, categorical_features)

numeric_summary

,pct_missing,pct_zero,pct_mode,mode,mean,std,iqr,cv,n_unique
NatWalkInd,18.291492,0.000000,2.227533,16.000000,12.550596,4.125104,6.666667,0.328678,114
D3A,18.291492,0.000000,0.128019,27.524029,21.927454,12.298200,17.056406,0.560859,6207
D1C,18.291492,0.146444,0.179227,0.000000,28.552549,136.320310,10.901872,4.774366,6192
D1B,18.291492,0.216179,0.264573,0.000000,40.634966,63.117383,53.089343,1.553278,6186
median_household_income,10.425384,0.000000,1.370183,250001.000000,92797.537797,47439.949267,58292.000000,0.511220,5833
family_households_proportion,1.317992,0.160391,0.847997,1.000000,0.612593,0.194444,0.265888,0.317411,6761
no_vehicle_households_proportion,1.317992,11.938633,12.098085,0.000000,0.261463,0.280915,0.399937,1.074395,5922
poverty_rate,1.317992,5.885635,5.964243,0.000000,0.143148,0.140365,0.156180,0.980558,6685
owner_occupied_proportion,1.317992,5.348675,5.420112,0.000000,0.536239,0.317086,0.570353,0.591314,6339
unemployment_rate,1.241283,17.663877,17.885892,0.000000,0.065370,0.074248,0.075613,1.135809,5357


In [23]:
categorical_summary['chain']

{'pct_missing': np.float64(0.0),
 'distribution':                    pct
 chain                 
 independent  73.584379
 chain        26.415621}

## Correlation

In [26]:
correlation_file_path = Path(".").resolve().parent / "src/correlation_test/high_correlation_test.csv"
correlation_df = pd.read_csv(str(correlation_file_path))

correlation_df

,variable_1,variable_2,correlation,abs_correlation
0,commercial_places_within_1_mi,community_places_within_1_mi,0.955,0.955
1,wac_cns12,wac_c000,0.845,0.845
2,wac_cns19,wac_c000,0.838,0.838
3,competitors_within_1_mi,no_vehicle_households_proportion,0.804,0.804
4,total_population,rac_c000,0.801,0.801
5,wac_cns10,wac_cns11,0.798,0.798
6,wac_cns10,wac_cns12,0.790,0.790
7,wac_cns12,wac_cns19,0.774,0.774
8,owner_occupied_proportion,no_vehicle_households_proportion,-0.757,0.757
9,wac_cns14,wac_cns19,0.742,0.742


## PCA Data

In [31]:
pca_train_file_path = Path(".").resolve().parent / "data/modeling/pca/X_pca_train.parquet"
pca_df = pd.read_parquet(str(pca_train_file_path))

print(pca_df.shape)
print(pca_df.columns)
pca_df.sample(5)

(8669, 18)
Index(['PC1', 'PC2', 'PC3', 'PC4', 'PC5', 'PC6', 'PC7', 'PC8', 'PC9', 'PC10',
       'PC11', 'PC12', 'PC13', 'PC14', 'PC15', 'PC16', 'PC17', 'agtno'],
      dtype='str')


,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,PC11,PC12,PC13,PC14,PC15,PC16,PC17,agtno
2814,0.028394,0.395168,1.911178,-0.551681,-0.472338,0.680768,-1.222844,0.794011,-0.388497,-1.961498,-1.660499,-0.025217,0.604225,0.535091,0.049790,-0.051998,0.284943,112575
1230,-2.952333,-0.118574,-0.793954,-1.846201,1.177825,0.812116,1.188990,-0.177347,-0.158441,-0.518332,1.086651,0.559186,0.557783,0.167826,0.407860,0.299042,-0.471628,107149
124,1.075009,-1.353118,-1.341559,1.167213,-1.242210,-2.013974,0.250436,-0.753897,-0.602246,0.420516,0.478473,0.048874,-0.392518,-0.021192,0.015550,-0.679831,-0.027195,100933
1893,0.544421,0.604836,0.977061,-2.482978,0.403044,0.853594,0.277180,-0.624402,-0.359190,0.059725,-0.106887,0.048689,-0.034018,0.030917,0.152526,0.517181,0.254937,109785
4163,3.707824,-0.904453,-0.187596,1.959834,1.634690,-0.965248,0.672404,0.038969,-0.173099,0.672912,-0.214933,-0.372717,0.566935,-0.120577,-0.104593,0.175620,0.074562,11561


## GMM Data

In [32]:
gmm_train_file_path = Path(".").resolve().parent / "data/modeling/gmm/X_gmm_train.parquet"
gmm_df = pd.read_parquet(str(gmm_train_file_path))

print(gmm_df.shape)
print(gmm_df.columns)
gmm_df.sample(5)

(8669, 8)
Index(['agtno', 'gmm_prob_0', 'gmm_prob_1', 'gmm_prob_2', 'gmm_prob_3',
       'gmm_prob_4', 'gmm_cluster', 'gmm_max_prob'],
      dtype='str')


,agtno,gmm_prob_0,gmm_prob_1,gmm_prob_2,gmm_prob_3,gmm_prob_4,gmm_cluster,gmm_max_prob
250,101655,0.933477,0.000165,0.026696,0.000400,0.039263,0,0.933477
5166,36978,0.612369,0.006123,0.050360,0.009674,0.321474,0,0.612369
8546,98145,0.475646,0.000285,0.069973,0.000028,0.454068,0,0.475646
2554,111879,0.097953,0.023440,0.000151,0.876582,0.001874,3,0.876582
2510,111706,0.010440,0.163098,0.000785,0.777876,0.047800,3,0.777876
